# 2.3 · 传输层：stdio 与 Streamable HTTP

**Transports: stdio and Streamable HTTP**

> **学完本课你能：** 在 stdio 和 Streamable HTTP 上正确地为 MCP 消息分帧，设置并校验必需的 HTTP 请求头，保护本地 HTTP 服务器，并处理取消和旧版传输。

官方阅读材料：[Spec: Transports overview](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/index) · [Spec: stdio](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/stdio) · [Spec: Streamable HTTP](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports/streamable-http)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 手动实现 stdio 分帧：流水线、stderr 与损坏的帧
一个在工作时把日志写到 <strong>stderr</strong> 的服务器（Server）。我们在一次写入中以流水线方式发送两个请求，观察答复<strong>按 id、乱序</strong>到达，并分别读取 stdout 和 stderr。

In [ ]:
import json
import queue
import subprocess
import sys
import tempfile
import textwrap
import threading
import time
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "stdio_server.py").write_text(textwrap.dedent('''
    import sys
    import anyio
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context

    mcp = MCPServer("stdio-lab", log_level="ERROR")

    @mcp.tool()
    def add(a: int, b: int) -> int:
        """Add two integers."""
        return a + b

    @mcp.tool()
    async def slow(steps: int, ctx: Context) -> str:
        """Work for a while, reporting progress."""
        for i in range(steps):
            print(f"[server log] slow step {i + 1}/{steps}", file=sys.stderr, flush=True)   # stderr: fine
            await ctx.report_progress(i + 1, steps)
            await anyio.sleep(0.3)
        return "done"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))

ENVELOPE = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientCapabilities": {}}
proc = subprocess.Popen([sys.executable, str(workdir / "stdio_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
stdout_q, stderr_q = queue.Queue(), queue.Queue()
threading.Thread(target=lambda: [stdout_q.put(l) for l in proc.stdout], daemon=True).start()
threading.Thread(target=lambda: [stderr_q.put(l) for l in proc.stderr], daemon=True).start()


def write_lines(*messages):
    data = "".join(json.dumps(m) + "\n" for m in messages)     # compact JSON, one per line
    proc.stdin.write(data)
    proc.stdin.flush()


def read_stdout(timeout=3.0):
    try:
        return json.loads(stdout_q.get(timeout=timeout))
    except queue.Empty:
        return None


write_lines(
    {"jsonrpc": "2.0", "id": "slow-1", "method": "tools/call",
     "params": {"name": "slow", "arguments": {"steps": 3}, "_meta": {**ENVELOPE, "progressToken": "p"}}},
    {"jsonrpc": "2.0", "id": "fast-2", "method": "tools/call",
     "params": {"name": "add", "arguments": {"a": 2, "b": 40}, "_meta": ENVELOPE}},
)
start = time.time()
for _ in range(5):
    msg = read_stdout(timeout=10)
    label = f"response id={msg['id']}" if "id" in msg else f"{msg['method']} {msg['params'].get('progress')}"
    print(f"stdout +{time.time() - start:.2f}s  {label}")
time.sleep(0.1)
while not stderr_q.empty():
    print("stderr          ", stderr_q.get().rstrip())

`fast-2` 是第二个写入的，却最先得到答复。客户端（Client）按 `id` 而不是按顺序匹配答复。现在来破坏分帧（Framing）：把同一个请求<strong>美化打印</strong>后发送（带内嵌换行）。

In [ ]:
pretty = json.dumps({"jsonrpc": "2.0", "id": 3, "method": "tools/list", "params": {"_meta": ENVELOPE}}, indent=2)
print(f"pretty-printed message spans {pretty.count(chr(10)) + 1} lines")
proc.stdin.write(pretty + "\n")
proc.stdin.flush()
print("reply:", read_stdout(timeout=2))

write_lines({"jsonrpc": "2.0", "id": 4, "method": "tools/list", "params": {"_meta": ENVELOPE}})
print("compact reply has", len(read_stdout()["result"]["tools"]), "tools - the server survived, but request 3 was lost")

不带 `indent` 的 `json.dumps` 永远不会输出原始换行：字符串<em>内部</em>的换行会被转义为 `\n`，所以任意文本都可以安全地放在一帧里。

### 2. stdio 的取消与关停
用 `notifications/cancelled`（取消通知）取消一个正在运行的请求，然后通过关闭 stdin 优雅地关停（Graceful Shutdown）。

In [ ]:
write_lines({"jsonrpc": "2.0", "id": "long-5", "method": "tools/call",
             "params": {"name": "slow", "arguments": {"steps": 20}, "_meta": {**ENVELOPE, "progressToken": "q"}}})
print("first message:", read_stdout(timeout=10)["params"])
write_lines({"jsonrpc": "2.0", "method": "notifications/cancelled",
             "params": {"requestId": "long-5", "reason": "user pressed stop"}})
print("after cancel :", read_stdout(timeout=2), "<- no response is sent for a cancelled request")

t = time.time()
proc.stdin.close()                                   # step 1: close the input stream
code = proc.wait(timeout=5)                          # step 2: wait (step 3 would be terminate/kill)
print(f"server exited with code {code} {time.time() - t:.2f}s after EOF")
time.sleep(0.1)
log_lines = []
while not stderr_q.empty():
    log_lines.append(stderr_q.get().strip())
print("stderr shows the cancelled work stopped early:", [l for l in log_lines if "/20" in l])

### 3. Streamable HTTP：单一端点，原始 POST
在一个线程中于 `127.0.0.1` 上运行 HTTP 服务器。一个小的 ASGI 包装器会<strong>记录每个请求的请求头</strong>，让我们能确切看到客户端（Client）发送了什么。其中一个工具声明了 `x-mcp-header` 参数。

In [ ]:
import logging
import socket
from typing import Annotated

import anyio
import httpx
import uvicorn
from pydantic import Field
from mcp.server import MCPServer
from mcp.server.mcpserver import Context

web_server = MCPServer("http-lab")


@web_server.tool()
def run_sql(region: Annotated[str, Field(description="Region to run in", json_schema_extra={"x-mcp-header": "Region"})],
            query: str) -> str:
    """Run a (pretend) SQL query in a region."""
    return f"[{region}] {query} -> 42 rows"


@web_server.tool()
async def slow(steps: int, ctx: Context) -> str:
    """Work for a while, reporting progress."""
    try:
        for i in range(steps):
            await ctx.report_progress(i + 1, steps, f"step {i + 1}")
            await anyio.sleep(0.2)
    except BaseException as exc:            # cancellation arrives as an exception in the handler
        server_events.append(f"slow() interrupted by {type(exc).__name__}")
        raise
    return "done"


for noisy in ("httpx", "httpx2", "mcp", "uvicorn"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

server_events, seen_headers = [], []
mcp_app = web_server.streamable_http_app()          # binds DNS-rebinding protection for 127.0.0.1


async def recorder(scope, receive, send):
    if scope["type"] == "http":
        seen_headers.append({"method": scope["method"],
                             **{k.decode(): v.decode() for k, v in scope["headers"] if k.startswith(b"mcp")}})
    await mcp_app(scope, receive, send)


sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()
uv = uvicorn.Server(uvicorn.Config(recorder, host="127.0.0.1", port=port, log_level="warning", lifespan="on"))
threading.Thread(target=uv.run, daemon=True).start()
while not uv.started:
    time.sleep(0.05)
URL = f"http://127.0.0.1:{port}/mcp"
web = httpx.Client(trust_env=False, timeout=10)
HEADERS = {"Accept": "application/json, text/event-stream", "Content-Type": "application/json",
           "MCP-Protocol-Version": "2026-07-28"}


def tools_call(name, arguments, id_=1, meta=ENVELOPE):
    return {"jsonrpc": "2.0", "id": id_, "method": "tools/call",
            "params": {"name": name, "arguments": arguments, "_meta": meta}}


r = web.post(URL, json=tools_call("run_sql", {"region": "us-west1", "query": "select 1"}),
             headers={**HEADERS, "Mcp-Method": "tools/call", "Mcp-Name": "run_sql", "Mcp-Param-Region": "us-west1"})
print(r.status_code, r.headers["content-type"])
print(json.dumps(r.json(), indent=2)[:600])

### 4. JSON 还是 SSE？由请求决定
同一个 `slow` 工具（Tool），在没人要求进度时以 JSON 作答，在请求携带 `progressToken`（进度令牌（Progress Token））时以 SSE 流作答。这个 SDK 只有在处理函数真正发出通知时才会切换到 SSE。

In [ ]:
def parse_sse(text):
    return [json.loads(line[len("data: "):]) for line in text.splitlines() if line.startswith("data: ")]


h = {**HEADERS, "Mcp-Method": "tools/call", "Mcp-Name": "slow"}
r = web.post(URL, json=tools_call("slow", {"steps": 2}), headers=h)
print("no progressToken  ->", r.status_code, r.headers["content-type"])

r = web.post(URL, json=tools_call("slow", {"steps": 3}, meta={**ENVELOPE, "progressToken": "tok-1"}), headers=h)
print("with progressToken->", r.status_code, r.headers["content-type"],
      "| X-Accel-Buffering:", r.headers.get("x-accel-buffering"))
for event in parse_sse(r.text):
    print("   SSE event:", event.get("method") or f"final response id={event['id']}",
          event.get("params", {}).get("message", ""))

### 5. 请求头校验：把每个请求头都弄错
下面每个请求都恰好错在一处。除了最后两个以外，其他请求都会在任何工具运行<strong>之前</strong>被拒绝。

In [ ]:
good_body = tools_call("run_sql", {"region": "us-west1", "query": "select 1"})
base = {**HEADERS, "Mcp-Method": "tools/call", "Mcp-Name": "run_sql", "Mcp-Param-Region": "us-west1"}
cases = {
    "all correct": base,
    "Mcp-Method lies": {**base, "Mcp-Method": "tools/list"},
    "Mcp-Name lies (WAF bypass attempt)": {**base, "Mcp-Name": "get_weather"},
    "Mcp-Name missing": {k: v for k, v in base.items() if k != "Mcp-Name"},
    "Mcp-Param-Region lies": {**base, "Mcp-Param-Region": "eu-west1"},
    "Mcp-Param-Region missing": {k: v for k, v in base.items() if k != "Mcp-Param-Region"},
    "Accept lacks text/event-stream": {**base, "Accept": "application/json"},
    "header NAMES are case-insensitive": {k.upper(): v for k, v in base.items()},
}
old_body = tools_call("run_sql", {"region": "us-west1", "query": "select 1"},
                      meta={**ENVELOPE, "io.modelcontextprotocol/protocolVersion": "2025-11-25"})
for label, headers in cases.items():
    r = web.post(URL, json=good_body, headers=headers)
    detail = r.json().get("error", {}) if r.headers.get("content-type", "").startswith("application/json") else {}
    print(f"{label:<36} HTTP {r.status_code}  {detail.get('code', '')} {detail.get('message', '')[:70]}")

r = web.post(URL, json=old_body, headers=base)          # header says 2026-07-28, body says 2025-11-25
print(f"{'body version != header':<36} HTTP {r.status_code}  {r.json()['error']['code']} {r.json()['error']['message'][:70]}")

`MCP-Protocol-Version` 检查在受支持版本检查之前运行，所以自相矛盾的客户端收到的是 `-32020`，而不是 `-32022`。有一个特殊之处：这个双时代（Dual-era）SDK 根据请求头来选择<em>时代</em>，所以<strong>请求头</strong>中写着旧版本的请求会被交给旧版传输处理（你会在第 8 节看到）。

非 ASCII 值以 base64 哨兵形式传输。SDK 提供了与客户端所用相同的编码器：

In [ ]:
from mcp.shared.inbound import encode_header_value

for value in ["us-west1", "Zürich", " padded ", "=?base64?literal?="]:
    print(f"{value!r:<22} -> Mcp-Param-Region: {encode_header_value(value)}")

r = web.post(URL, json=tools_call("run_sql", {"region": "Zürich", "query": "select 1"}),
             headers={**base, "Mcp-Param-Region": encode_header_value("Zürich")})
print("\nencoded header accepted:", r.status_code, r.json()["result"]["content"][0]["text"])

### 6. 用 Origin 和 Host 检查防御 DNS 重绑定
由于该应用是为主机 `127.0.0.1` 构建的，SDK 启用了 DNS 重绑定（DNS Rebinding）防护，并使用只包含 localhost 的白名单。`evil.example` 上把自己的名字重绑定到 127.0.0.1 的页面，仍然会发送它自己的 `Origin` 和 `Host`：

In [ ]:
list_body = {"jsonrpc": "2.0", "id": 9, "method": "tools/list", "params": {"_meta": ENVELOPE}}
lh = {**HEADERS, "Mcp-Method": "tools/list"}
for label, extra in [("no Origin (non-browser client)", {}),
                     ("Origin http://localhost:5173", {"Origin": "http://localhost:5173"}),
                     ("Origin http://evil.example", {"Origin": "http://evil.example"}),
                     ("Host evil.example (rebinding)", {"Host": f"evil.example:{port}"})]:
    r = web.post(URL, json=list_body, headers={**lh, **extra})
    print(f"{label:<32} -> HTTP {r.status_code} {r.text[:40] if r.status_code != 200 else ''}")

### 7. 通过关闭流来取消
启动一个流式发送进度的长调用，读取两个事件，然后<strong>关闭连接</strong>。不会发送 `notifications/cancelled`：断开连接本身就是取消（Cancellation）信号。

In [ ]:
server_events.clear()
body = tools_call("slow", {"steps": 30}, meta={**ENVELOPE, "progressToken": "tok-2"})
with web.stream("POST", URL, json=body, headers={**HEADERS, "Mcp-Method": "tools/call", "Mcp-Name": "slow"}) as r:
    received = 0
    for line in r.iter_lines():
        if line.startswith("data: "):
            received += 1
            print("got:", json.loads(line[6:])["params"]["message"])
            if received == 2:
                break                    # leaving the block closes the response stream
time.sleep(0.5)
print("server side:", server_events)

### 8. 旧客户端与高层 `Client`
先看 2026-07-28 服务器如何对待旧版流量，再看 SDK 的 `Client` 自己会发送什么。

In [ ]:
print("GET  /mcp                ->", web.get(URL, headers=HEADERS).status_code, "(no standalone SSE stream any more)")
print("DELETE /mcp              ->", web.delete(URL, headers=HEADERS).status_code, "(no sessions to delete)")
note = {"jsonrpc": "2.0", "method": "notifications/cancelled", "params": {"requestId": 1}}
print("POST a notification      ->", web.post(URL, json=note, headers={**HEADERS, "Mcp-Method": "notifications/cancelled"}).status_code)
r = web.post(URL, json=list_body, headers={k: v for k, v in lh.items() if k != "MCP-Protocol-Version"})
print("POST without version hdr ->", r.status_code, r.json()["error"]["message"],
      "\n   (this SDK is dual-era: no header = an old client, so the legacy session transport answered)")

最后是真正的客户端（Client）。它会进行发现、发送 `Mcp-Method`/`Mcp-Name`，并镜像 `x-mcp-header` 参数。如果它还不知道模式（Schema），服务器返回的 `-32020` 会让它重新列出工具并重试，完全符合规范的建议。

In [ ]:
from mcp import Client

seen_headers.clear()


async def on_progress(progress, total, message):
    print(f"   progress {progress:.0f}/{total:.0f} {message}")


async with Client(URL) as client:
    print("negotiated:", client.protocol_version)
    print((await client.call_tool("run_sql", {"region": "Zürich", "query": "select 1"})).content[0].text)
    print((await client.call_tool("slow", {"steps": 2}, progress_callback=on_progress)).content[0].text)

print("\nwhat went over the wire:")
for h in seen_headers:
    print("  ", h)
uv.should_exit = True

### 动手练习
1. 在第 1 节服务器的 `add` 工具（Tool）里加一句 `print("hello")`（输出到 <strong>stdout</strong>），重启服务器并调用 `add`。原始客户端看到了什么？`Client` 又会怎么做？然后用 `file=sys.stderr` 修复它。
2. 在一个 `number` 参数上，或在数组 `items` 内部的某个属性上加 `x-mcp-header`，然后从 `Client(URL)` 调用 `list_tools()`。规范说 HTTP 客户端必须丢弃这样的工具：这个 SDK 会这样做吗？
3. 编写一个函数 `cancel_after(seconds)`，以流式方式发起 `slow` 调用，并在超出时间预算后关闭它。把服务器端的事件与第 2 节的 stdio 取消进行比较。在每种情况下，哪一方知道取消的原因？
4. 用 `web_server.streamable_http_app(json_response=True)` 重新构建 HTTP 应用。进度通知会怎样？为什么部署在会缓冲的代理后面的服务可能仍然会选择这样做？